In [ ]:
from __future__ import annotations

import json
import math
import re
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import numpy as np

from pydantic import BaseModel, Field

from sqlalchemy import text
from sqlalchemy.orm import Session

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder,
)

from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

from db.session import SessionLocal
from db.full_model import RagChunkORM

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

reranker = CrossEncoder(
    RERANKER_MODEL_NAME
)

e:\Thesis_all\be\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
C:\Users\AN515-56\AppData\Local\Temp\ipykernel_21928\1116921045.py:23: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


ModuleNotFoundError: No module named 'db'

In [ ]:
class QueryLocation(BaseModel):
    country: Optional[str] = None
    city: Optional[str] = None
    province: Optional[str] = None


class QueryConstraints(BaseModel):
    budget: Optional[str] = None
    duration_days: Optional[int] = None

    date_from: Optional[str] = None
    date_to: Optional[str] = None

    near_place: Optional[str] = None
    max_distance_km: Optional[float] = None


class ParsedQuery(BaseModel):
    intent: Optional[str] = None

    location: QueryLocation = Field(
        default_factory=QueryLocation
    )

    place_types: list[str] = Field(
        default_factory=list
    )

    activities: list[str] = Field(
        default_factory=list
    )

    travel_styles: list[str] = Field(
        default_factory=list
    )

    suitable_for: list[str] = Field(
        default_factory=list
    )

    constraints: QueryConstraints = Field(
        default_factory=QueryConstraints
    )

In [ ]:
class UserTravelMemory(BaseModel):
    preferred_travel_styles: list[str] = Field(
        default_factory=list
    )

    preferred_activities: list[str] = Field(
        default_factory=list
    )

    budget_level: Optional[str] = None

    avoid: list[str] = Field(
        default_factory=list
    )

In [ ]:
class UserTravelMemory(BaseModel):
    preferred_travel_styles: list[str] = Field(
        default_factory=list
    )

    preferred_activities: list[str] = Field(
        default_factory=list
    )

    budget_level: Optional[str] = None

    avoid: list[str] = Field(
        default_factory=list
    )

In [ ]:
query = """
Plan family-friendly activities in Da Nang for 3 days.
I like cultural attractions and local food.
"""

conversation_history = [
    {
        "role": "user",
        "content": "I am planning a Vietnam trip."
    }
]

In [ ]:
def rewrite_query(
    query: str,
    conversation_history: list[dict],
) -> str:

    if not conversation_history:
        return query.strip()

    history_text = "\n".join(
        f"{item['role']}: {item['content']}"
        for item in conversation_history[-5:]
    )

    # Later replace this with DeepSeek.
    #
    # For now simply return query so you can test
    # the rest of retrieval independently.

    return query.strip()

In [ ]:
rewritten_query = rewrite_query(
    query=query,
    conversation_history=conversation_history,
)

print(rewritten_query)

# SEMATIC QUERY parsing

In [ ]:
CITY_ALIASES = {
    "da nang": "Da Nang",
    "danang": "Da Nang",

    "hanoi": "Hanoi",
    "ha noi": "Hanoi",

    "hoi an": "Hoi An",

    "hue": "Hue",

    "saigon": "Ho Chi Minh City",
    "sai gon": "Ho Chi Minh City",
    "ho chi minh": "Ho Chi Minh City",
    "ho chi minh city": "Ho Chi Minh City",

    "ha long": "Ha Long",
}


TRAVEL_STYLE_KEYWORDS = {
    "family": "family",
    "family-friendly": "family",
    "luxury": "luxury",
    "adventure": "adventure",
    "cultural": "culture",
    "culture": "culture",
    "budget": "budget",
}


ACTIVITY_KEYWORDS = {
    "shopping": "shopping",
    "hiking": "hiking",
    "swimming": "swimming",
    "food": "food_tasting",
    "street food": "street_food",
    "sightseeing": "sightseeing",
    "museum": "museum_visit",
}

In [ ]:
def parse_query(query: str) -> ParsedQuery:
    lower = query.lower()
    parsed = ParsedQuery()
    # --------------------------
    # LOCATION
    # --------------------------
    for alias, canonical in CITY_ALIASES.items():

        if alias in lower:
            parsed.location.city = canonical
            parsed.location.country = "Vietnam"
            break
    # --------------------------
    # TRAVEL STYLE
    # --------------------------
    for keyword, style in TRAVEL_STYLE_KEYWORDS.items():

        if keyword in lower:
            parsed.travel_styles.append(style)

    parsed.travel_styles = list(
        dict.fromkeys(parsed.travel_styles)
    )
    # --------------------------
    # ACTIVITIES
    # --------------------------
    for keyword, activity in ACTIVITY_KEYWORDS.items():

        if keyword in lower:
            parsed.activities.append(activity)

    parsed.activities = list(
        dict.fromkeys(parsed.activities)
    )
    # --------------------------
    # SUITABLE FOR
    # --------------------------
    if "family" in lower or "kids" in lower:
        parsed.suitable_for.append(
            "families"
        )

    # --------------------------
    # DURATION
    # --------------------------
    match = re.search(
        r"(\d+)\s*days?",
        lower,
    )

    if match:
        parsed.constraints.duration_days = int(
            match.group(1)
        )

    # --------------------------
    # INTENT
    # --------------------------

    if (
        "plan" in lower
        and parsed.constraints.duration_days
    ):
        parsed.intent = "itinerary"

    elif "recommend" in lower:
        parsed.intent = "recommendation"

    else:
        parsed.intent = "travel_information"

    return parsed

In [ ]:
def parse_query(
    query: str,
) -> ParsedQuery:

    lower = query.lower()

    parsed = ParsedQuery()

    # --------------------------
    # LOCATION
    # --------------------------

    for alias, canonical in CITY_ALIASES.items():

        if alias in lower:
            parsed.location.city = canonical
            parsed.location.country = "Vietnam"
            break

    # --------------------------
    # TRAVEL STYLE
    # --------------------------

    for keyword, style in TRAVEL_STYLE_KEYWORDS.items():

        if keyword in lower:
            parsed.travel_styles.append(style)

    parsed.travel_styles = list(
        dict.fromkeys(parsed.travel_styles)
    )

    # --------------------------
    # ACTIVITIES
    # --------------------------

    for keyword, activity in ACTIVITY_KEYWORDS.items():

        if keyword in lower:
            parsed.activities.append(activity)

    parsed.activities = list(
        dict.fromkeys(parsed.activities)
    )

    # --------------------------
    # SUITABLE FOR
    # --------------------------

    if "family" in lower or "kids" in lower:
        parsed.suitable_for.append(
            "families"
        )

    # --------------------------
    # DURATION
    # --------------------------

    match = re.search(
        r"(\d+)\s*days?",
        lower,
    )

    if match:
        parsed.constraints.duration_days = int(
            match.group(1)
        )

    # --------------------------
    # INTENT
    # --------------------------

    if (
        "plan" in lower
        and parsed.constraints.duration_days
    ):
        parsed.intent = "itinerary"

    elif "recommend" in lower:
        parsed.intent = "recommendation"

    else:
        parsed.intent = "travel_information"

    return parsed

# USER memory

In [ ]:
def get_user_memory(
    user_id: Optional[str],
) -> UserTravelMemory:

    if not user_id:
        return UserTravelMemory()

    # Replace later with DB query.

    return UserTravelMemory(
        preferred_travel_styles=[
            "culture",
            "food",
        ],
        preferred_activities=[
            "sightseeing",
        ],
        budget_level="mid_range",
    )

In [ ]:
user_memory = get_user_memory(
    user_id="test-user"
)

user_memory.model_dump()